In [4]:
%%capture
import os, importlib.util
!pip install --upgrade -qqq uv
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try: import numpy, PIL; _numpy = f"numpy=={numpy.__version__}"; _pil = f"pillow=={PIL.__version__}"
    except: _numpy = "numpy"; _pil = "pillow"
    !uv pip install -qqq \
        "torch==2.8.0" "triton>=3.3.0" {_numpy} {_pil} torchvision bitsandbytes xformers==0.0.32.post2 \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth"
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth
!uv pip install --upgrade --no-deps tokenizers trl==0.22.2 unsloth unsloth_zoo
!uv pip install transformers==5.2.0
# causal_conv1d is supported only on torch==2.8.0. If you have newer torch versions, please wait 10 minutes!
!uv pip install --no-build-isolation flash-linear-attention causal_conv1d==1.6.0

In [1]:
!pip install unsloth trl transformers datasets accelerate bitsandbytes wandb

In [2]:
import os
import random
import logging
from datasets import load_dataset, concatenate_datasets, Dataset

# logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
log = logging.getLogger(__name__)


In [3]:
MODEL_NAME = "Qwen/Qwen3-0.8B"  # base model (unsloth will auto-find 4bit version)
MAX_SEQ_LENGTH = 4096
LORA_RANK = 16
BATCH_SIZE= 2
GRAD_ACCUM= 8    # effective batch = 2 * 8 = 16
LEARNING_RATE= 2e-4
EPOCHS= 3
OUTPUT_DIR= "/outputs/qwen3-0.8b-research-agent"
SEED= 42

WEIGHTS = {
    "instruction":   0.35,
    "summarization": 0.30,
    "extraction":    0.20,
    "decomposition": 0.15,
}
TARGET_TOTAL = 100_000   # total samples in final combined dataset


In [ ]:


# ─────────────────────────────────────────────────────────────────────────────
# STEP 1 — LOAD DATASETS
# ─────────────────────────────────────────────────────────────────────────────
def load_all_datasets():
    log.info("Loading datasets from HuggingFace Hub ...")
    ds1 = load_dataset("Rohit-Katkar2003/Instruction-dataset",   split="train")
    ds2 = load_dataset("Rohit-Katkar2003/summarization-dataset", split="train")
    ds3 = load_dataset("Rohit-Katkar2003/Extractional-dataset",  split="train")
    ds4 = load_dataset("Rohit-Katkar2003/decomposition-dataset", split="train")
    log.info(f"  D1 Instruction:   {len(ds1):>7,} samples")
    log.info(f"  D2 Summarization: {len(ds2):>7,} samples")
    log.info(f"  D3 Extraction:    {len(ds3):>7,} samples")
    log.info(f"  D4 Decomposition: {len(ds4):>7,} samples")
    return ds1, ds2, ds3, ds4


# ─────────────────────────────────────────────────────────────────────────────
# STEP 2 — WEIGHTED SAMPLING + COMBINE
# ─────────────────────────────────────────────────────────────────────────────
def weighted_combine(ds1, ds2, ds3, ds4) -> Dataset:
    log.info("Combining datasets with weighted sampling ...")
    random.seed(SEED)

    source_map = {
        "instruction":   ds1,
        "summarization": ds2,
        "extraction":    ds3,
        "decomposition": ds4,
    }

    selected_splits = []
    for name, ds in source_map.items():
        target_n = int(TARGET_TOTAL * WEIGHTS[name])

        if len(ds) >= target_n:
            # Downsample
            idx = random.sample(range(len(ds)), target_n)
            split = ds.select(idx)
        else:
            # Upsample with repetition (important for small D4)
            repeats = (target_n // len(ds)) + 1
            idx = (list(range(len(ds))) * repeats)[:target_n]
            random.shuffle(idx)
            split = ds.select(idx)
            log.info(f"  ↑ Upsampled {name}: {len(ds):,} → {target_n:,}")

        selected_splits.append(split)
        log.info(f"  {name:<15} → {target_n:,} samples ({WEIGHTS[name]*100:.0f}%)")

    # Concatenate all splits
    combined = concatenate_datasets(selected_splits)

    # Shuffle the final combined dataset
    combined = combined.shuffle(seed=SEED)
    log.info(f"  ✅ Combined total: {len(combined):,} samples")
    return combined


# ─────────────────────────────────────────────────────────────────────────────
# STEP 3 — FORMAT TO CHATML TEXT FOR UNSLOTH
# ─────────────────────────────────────────────────────────────────────────────
def format_to_chatml(sample: dict) -> dict:
    """
    Convert internal format → ChatML text string for SFT training.

    Input schema:
      system:   str
      messages: list of {role: str, content: str}

    Output:
      text: full ChatML string Unsloth can train on
    """
    system   = sample.get("system", "You are a helpful AI assistant.").strip()
    messages = sample.get("messages", [])

    parts = [f"<|im_start|>system\n{system}<|im_end|>"]

    for msg in messages:
        role    = msg.get("role", "").strip()
        content = msg.get("content", "").strip()
        if role in ("user", "assistant") and content:
            parts.append(f"<|im_start|>{role}\n{content}<|im_end|>")

    # Qwen3 expects a trailing newline
    text = "\n".join(parts) + "\n"

    return {"text": text}


def apply_formatting(combined: Dataset) -> Dataset:
    log.info("Formatting dataset to ChatML ...")
    formatted = combined.map(
        format_to_chatml,
        remove_columns=combined.column_names,   # drop all old columns, keep only "text"
        num_proc=4,
        desc="Formatting to ChatML",
    )

    # Filter out sequences that are too short (likely broken examples)
    before = len(formatted)
    formatted = formatted.filter(
        lambda x: len(x["text"]) > 50,
        desc="Filtering short examples",
    )
    log.info(f"  Filtered {before - len(formatted)} short examples")
    log.info(f"  ✅ Final formatted size: {len(formatted):,} samples")
    return formatted


# ─────────────────────────────────────────────────────────────────────────────
# STEP 4 — TRAIN/VAL SPLIT
# ─────────────────────────────────────────────────────────────────────────────
def split_dataset(formatted: Dataset):
    split = formatted.train_test_split(test_size=0.05, seed=SEED)
    log.info(f"  Train: {len(split['train']):,}  |  Val: {len(split['test']):,}")
    return split["train"], split["test"]



: 

In [5]:
def load_model():
    log.info(f"Loading model: {MODEL_NAME} with Unsloth ...")
    from unsloth import FastLanguageModel
    import torch

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name      = MODEL_NAME,
        max_seq_length  = MAX_SEQ_LENGTH,
        dtype           = None,         # auto: bfloat16 on Ampere+, float16 on older
        load_in_4bit    = True,         # QLoRA 4-bit
    )

    # Apply LoRA adapters
    model = FastLanguageModel.get_peft_model(
        model,
        r               = LORA_RANK,
        target_modules  = [
            "q_proj", "k_proj", "v_proj", "o_proj",    # Attention
            "gate_proj", "up_proj", "down_proj",         # MLP
        ],
        lora_alpha      = LORA_RANK * 2,    # Standard: 2x rank
        lora_dropout    = 0.05,
        bias            = "none",
        use_gradient_checkpointing = "unsloth",   # Saves ~30% VRAM
        random_state    = SEED,
        use_rslora      = True,             # Rank-stabilized LoRA (better convergence)
    )

    log.info("Model loaded with LoRA adapters ✅")
    log.info(f"  {model.print_trainable_parameters()}")
    return model, tokenizer


# ─────────────────────────────────────────────────────────────────────────────
# STEP 6 — TRAIN
# ─────────────────────────────────────────────────────────────────────────────
def train(model, tokenizer, train_ds, val_ds):
    from trl import SFTTrainer, SFTConfig

    os.makedirs(OUTPUT_DIR, exist_ok=True)

    training_args = SFTConfig(
        output_dir                  = OUTPUT_DIR,

        # Epochs & batching
        num_train_epochs            = EPOCHS,
        per_device_train_batch_size = BATCH_SIZE,
        per_device_eval_batch_size  = BATCH_SIZE,
        gradient_accumulation_steps = GRAD_ACCUM,

        # Learning rate
        learning_rate               = LEARNING_RATE,
        warmup_ratio                = 0.03,
        lr_scheduler_type           = "cosine",
        weight_decay                = 0.01,

        # Precision
        bf16                        = True,
        fp16                        = False,

        # Sequence packing (fills context window, faster training)
        packing                     = True,
        max_seq_length              = MAX_SEQ_LENGTH,
        dataset_text_field          = "text",

        # Eval & saving
        eval_strategy               = "steps",
        eval_steps                  = 200,
        save_strategy               = "steps",
        save_steps                  = 200,
        save_total_limit            = 3,
        load_best_model_at_end      = True,
        metric_for_best_model       = "eval_loss",
        greater_is_better           = False,

        # Logging
        logging_steps               = 50,
        report_to                   = "wandb",
        run_name                    = "qwen3-0.8b-research-agent",

        # Misc
        seed                        = SEED,
        dataloader_num_workers      = 2,
        optim                       = "adamw_8bit",   # memory-efficient optimizer
    )

    trainer = SFTTrainer(
        model           = model,
        tokenizer       = tokenizer,
        train_dataset   = train_ds,
        eval_dataset    = val_ds,
        args            = training_args,
    )

    log.info("Starting training ...")
    trainer_stats = trainer.train()
    log.info(f"Training complete: {trainer_stats}")

    return trainer


# ─────────────────────────────────────────────────────────────────────────────
# STEP 7 — SAVE
# ─────────────────────────────────────────────────────────────────────────────
def save_model(model, tokenizer):
    from unsloth import FastLanguageModel

    # Save LoRA adapters (small, ~50MB)
    adapter_path = f"{OUTPUT_DIR}/lora-adapters"
    model.save_pretrained(adapter_path)
    tokenizer.save_pretrained(adapter_path)
    log.info(f"LoRA adapters saved → {adapter_path}")

    # Save merged model in 16-bit (full model, ready for vLLM/llama.cpp)
    merged_path = f"{OUTPUT_DIR}/merged-16bit"
    model.save_pretrained_merged(merged_path, tokenizer, save_method="merged_16bit")
    log.info(f"Merged 16-bit model saved → {merged_path}")

    # Optional: Save as GGUF for llama.cpp / Ollama deployment
    # gguf_path = f"{OUTPUT_DIR}/gguf-q4"
    # model.save_pretrained_gguf(gguf_path, tokenizer, quantization_method="q4_k_m")
    # log.info(f"GGUF model saved → {gguf_path}")


# ─────────────────────────────────────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────────────────────────────────────
def main():
    os.environ.setdefault("WANDB_PROJECT", "research-agent-finetuning")

    # 1. Load
    ds1, ds2, ds3, ds4 = load_all_datasets()

    # 2. Weighted combine
    combined = weighted_combine(ds1, ds2, ds3, ds4)

    # 3. Format to ChatML
    formatted = apply_formatting(combined)

    # 4. Split
    train_ds, val_ds = split_dataset(formatted)

    # 5. Load model
    model, tokenizer = load_model()

    # 6. Train
    trainer = train(model, tokenizer, train_ds, val_ds)

    # 7. Save
    save_model(model, tokenizer)

    log.info("=" * 60)
    log.info("✅ All done! Model saved to: " + OUTPUT_DIR)
    log.info("=" * 60)


main()

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


: 

: 